In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
import tqdm
import json 
import copy
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from matplotlib import colors
import torchvision
import torch.nn.functional as F
import pandas as pd
from matplotlib.colors import ListedColormap
from typing import Dict, Tuple
from torchmetrics.functional import jaccard_index, precision, recall, f1_score 
from torchmetrics.functional.classification import binary_precision_recall_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, PrecisionRecallDisplay, roc_auc_score
from torchmetrics.classification import PrecisionRecallCurve
from torchmetrics import MeanAbsoluteError, MeanSquaredError, R2Score, PearsonCorrCoef, MetricCollection

from src.utils import config_helper, get_wandb_runs
from src.data.get_dataloaders import get_dataloaders
from src.data.dataset import DeforestationForecastingDataset
from src.algorithm.models.get_model import  get_model
from src.algorithm.loss import get_loss_fn
from src.evaluate import evaluate

In [3]:
import torch
import numpy as np
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn.functional as F

def reverse_sigmoid(x):
    # inverse of sigmoid (logit)
    eps = 1e-7  # to avoid division by zero
    x = torch.clamp(x, eps, 1 - eps)
    return torch.log(x / (1 - x))

class ExponentialDecayBaseline:
    """
    Risk decays exponentially with distance to nearest deforestation (value=1).
    Output risk only for forest pixels (value=0). Other values left at 0.
    """
    def __init__(self, decay_distance: float = 3.0): #Value 3 is based on hyperparam search
        self.decay_distance_pixels = float(decay_distance)

    def predict(self, deforestation_map: torch.Tensor) -> torch.Tensor:
        #if isinstance(deforestation_map, dict):
        #    deforestation_map = deforestation_map['deforestation_map']
        
        
        # deforestation_map: (B,H,W) with {1=deforestation, 0=forest, -1=invalid}
        dm = deforestation_map.cpu().numpy()
        risk_maps = []

        for b in range(dm.shape[0]):
            is_deforestation = (dm[b] == 1)
            is_forest        = (dm[b] == 0)

            if is_deforestation.any():
                # distance to nearest deforestation pixel
                distance_to_defor = distance_transform_edt(~is_deforestation)
            else:
                distance_to_defor = np.full(dm[b].shape, np.inf, dtype=float)

            risk = np.exp(-distance_to_defor / self.decay_distance_pixels)

            # keep risk only on forest pixels
            risk[~is_forest] = 0.0

            risk_maps.append(risk.astype(np.float32))

        #return reverse_sigmoid(torch.from_numpy(np.stack(risk_maps)).to('cuda'))
        return torch.from_numpy(np.stack(risk_maps))
    
    def eval(self):
        pass

    def __call__(self, deforestation_map):
        return self.predict(deforestation_map)

In [4]:
def split_inputs_for_multi_year(
    input_batch: Dict[str, torch.Tensor],
    T: int,
    K: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Prepare inputs and ground truth for a K-year evaluation, using a batch that already
    contains T+K time steps.

    Inputs
    ------
    input_batch : dict
        Must contain:
          - 'deforestation_map': Tensor (B, T+K, 1, H, W)
              Per-year encoding:
                1  -> deforested THIS year (first time)
                0  -> forest
               -1  -> non-forest (after the first deforestation year, all later years remain -1)
          - 'elevation': Tensor (B, 1, H, W) (static; the model repeats it over time internally)
    T : int
        Number of context years fed to the model.
    K : int
        Number of future years to forecast and evaluate.
    device : torch.device
        Device on which tensors will reside.

    Returns
    -------
    init_def_seq : Tensor (B, T, 1, H, W)
        The first T frames; used as the model’s input sequence.
    future_truth_seq : Tensor (K, B, H, W), dtype=bool
        Per-year ground truth for the next K years: True only in the first deforestation year.
        (Kept for convenience; evaluation below uses the full future frames to build masks precisely.)
    elevation : Tensor (B, 1, H, W)
        Static elevation map (unmodified).
    base_year_map : Tensor (B, H, W)
        The map at the base year t0 (last of the first T frames). Used to build evaluation masks.
    """
    def_seq_all = input_batch['deforestation_map'].to(device).to(torch.float32)  # (B, T+K, 1, H, W)
    elevation   = input_batch['elevation'].to(device).to(torch.float32)         # (B, 1, H, W)
    B, X, _, H, W = def_seq_all.shape
    assert X >= T + K, f"Expected at least T+K frames, got X={X} (T={T}, K={K})."

    # First T frames are the model input; next K frames are the future ground truth.
    init_def_seq = def_seq_all[:, :T]                         # (B, T, 1, H, W)
    future_frames = def_seq_all[:, T:T+K, 0]                  # (B, K, H, W)
    future_truth_seq = (future_frames == 1).permute(1, 0, 2, 3)  # (K, B, H, W) bool

    # Base year (t0) is the last frame among the first T.
    base_year_map = init_def_seq[:, -1, 0]                    # (B, H, W)

    return init_def_seq, future_truth_seq, elevation, base_year_map


@torch.no_grad()
def predict_deforestation_within_k_years(
    model: torch.nn.Module,
    init_def_seq: torch.Tensor,
    elevation: torch.Tensor,
    K: int,
    device: torch.device,
    binarize_feedback: bool = True,
    fixed_window: bool = True
) -> torch.Tensor:
    """
    Autoregressively predict whether each pixel will be marked as deforested at least once
    within the next K years. Returns a single binary cumulative prediction.

    Process
    -------
    We call the model K times. After each call we:
      1) Identify pixels newly marked as deforested this step by thresholding the predicted probabilities at 0.5,
         restricted to pixels that are still forest at this step.
      2) Mark those pixels as ineligible going forward (irreversibility).
      3) Construct the next input frame and append it to the time series.

    Important rules
    ---------------
    - A pixel that is already non-forest (-1) or already marked as deforested (1) in the base year (t0)
      is not eligible to be marked as deforested again in future steps.
    - If `binarize_feedback` is True, the appended frame uses the same encoding as training data:
        0 = forest, 1 = deforested THIS year, -1 = non-forest
      If False, we append probabilities on forest pixels and -1 on non-forest pixels (a soft-feedback experiment).
    - If `fixed_window` is True, we keep the input sequence length at T by dropping the oldest frame each step.
      If False, we let the time dimension grow.

    Returns
    -------
    Tensor (B, H, W), float32 in {0.0, 1.0}
        1.0 where the pixel is predicted to be deforested in ANY of the next K years (cumulative over steps),
        0.0 otherwise.
    """
    def_seq   = init_def_seq.clone().to(torch.float32).cpu()  # (B, T, 1, H, W)
    #elevation = elevation.clone().to(device).to(torch.float32)     # (B, 1, H, W)
    B, T, _, H, W = def_seq.shape

    # Pixels ineligible for future deforestation from the base year onward:
    #   - value -1 (already non-forest)
    #   - value  1 (already deforested at the base year)
    base_year_map = def_seq[:, -1, 0]                               # (B, H, W)
    ineligible_mask = (base_year_map == -1) | (base_year_map == 1)  # bool

    # Accumulator: did a pixel get predicted as deforested in ANY of the next K years?
    any_event_within_period = torch.zeros((B, H, W), device='cpu', dtype=torch.bool)

    for _ in range(K):
        # Predict probabilities for the "next year" relative to the current sequence.
        step_probs = model.predict(def_seq[:, -1].squeeze())

        #logits = model({'deforestation_map': def_seq, 'elevation': elevation})  # (B, T_current, 1, H, W)
        #step_probs = torch.sigmoid(logits[:, -1, 0])                            # (B, H, W), values in [0,1]

        # Only pixels that are still forest at this step are eligible to be newly marked as deforested.
        forest_mask = ~ineligible_mask

        # Threshold at 0.5 to decide which pixels are newly marked as deforested this step.
        new_event_mask = (step_probs >= 0.5) & forest_mask  # bool

        # Accumulate into the cumulative “within K years” prediction.
        any_event_within_period |= new_event_mask

        # Enforce irreversibility: once marked, a pixel remains ineligible for all later steps.
        ineligible_mask |= new_event_mask

        # Build the next frame to append to the sequence.
        if binarize_feedback:
            # Match training encoding: 0=forest, 1=deforested THIS year, -1=non-forest.
            next_frame = torch.zeros_like(base_year_map, dtype=torch.float32)  # default: forest (0)
            # All ineligible pixels (already non-forest OR became deforested this step) are -1,
            # then overwrite the current-step events with 1 to reflect "event this year".
            next_frame[ineligible_mask] = -1.0
            next_frame[new_event_mask]  = 1.0
        else:
            # Soft feedback: probabilities for forest pixels; -1 for ineligible pixels.
            next_frame = step_probs.clone()
            next_frame[ineligible_mask] = -1.0

        # Append the new frame and optionally drop the oldest to keep a fixed-length window.
        next_frame_t = next_frame.unsqueeze(1).unsqueeze(1)  # (B, 1, 1, H, W)
        if fixed_window:
            def_seq = torch.cat([def_seq[:, 1:], next_frame_t], dim=1)
        else:
            def_seq = torch.cat([def_seq, next_frame_t], dim=1)
        # Elevation remains (B,1,H,W); the model handles temporal repetition internally.

    return any_event_within_period.float()


def compute_metrics(targets, preds, dataset_name, regression_metrics=True):

    preds = preds.to(dtype=torch.float16)
    targets = targets.to(dtype=torch.int16)

    mask = targets != -1
    valid_targets = targets[mask].flatten()
    valid_preds = preds[mask].flatten()
    
    
    if regression_metrics:
        binarized_preds = (preds >= 0.5).int() 
        pred_pos = ((binarized_preds == 1) & mask).sum(dim=(1,2)).float()
        targ_pos = ((targets == 1) & mask).sum(dim=(1,2)).float()
        # stack for torchmetrics
        # exclude tiles with no valid pixels
        has_valid_data = mask.sum(dim=(1,2)) > 0
        pred_pos, targ_pos = pred_pos[has_valid_data], targ_pos[has_valid_data]
        pred_pos = pred_pos.unsqueeze(1)
        targ_pos = targ_pos.unsqueeze(1)
        reg_metrics_coll = MetricCollection({
            f'{dataset_name}_MAE': MeanAbsoluteError(),
            f'{dataset_name}_MSE': MeanSquaredError(),
            f'{dataset_name}_RMSE': MeanSquaredError(squared=False),
            f'{dataset_name}_R2': R2Score(),
            f'{dataset_name}_PearsonR': PearsonCorrCoef(),
        })

        reg_metrics = reg_metrics_coll(pred_pos, targ_pos)


    metrics = {
        f'{dataset_name}_iou' : jaccard_index(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_f1' : f1_score(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_precision' : precision(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_recall' : recall(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_average_precision' : average_precision_score(valid_targets.flatten(), valid_preds.flatten()),
        f'{dataset_name}_ROC_AUC' : roc_auc_score(valid_targets.flatten(), valid_preds.flatten()),
    }

    if regression_metrics:
        metrics.update(reg_metrics)


    return metrics


@torch.no_grad()
def evaluate_deforestation_within_k_years(
    model: torch.nn.Module,
    dataloader,
    device: torch.device,
    T: int,
    K: int,
    binarize_feedback: bool = True,
    fixed_window: bool = True,
    dataset_name: str = 'val',
    log_to_wandb: bool = False,
) -> Dict[str, torch.Tensor]:
    """
    Evaluate whether each pixel is correctly predicted to be deforested at least once
    within the next K years, using your existing dataloader (no changes to its API).

    Evaluation mask policy (handles -1 correctly across years)
    ----------------------------------------------------------
    A pixel is evaluated if:
      - it is forest in the base year t0 (value 0), AND
      - within the K-year period, it is never marked as -1 BEFORE its first deforestation year.
        (If no deforestation occurs within K, then any -1 in those K years causes the pixel to be ignored.)

    Steps per batch
    ---------------
      1) Split: first T frames -> model input; next K frames -> per-year ground truth (and statuses).
      2) Run K-step autoregressive prediction -> cumulative binary prediction (within K years).
      3) Build cumulative ground truth (deforested in any of the next K years) and the **validity mask** described above.
      4) Apply the validity mask: pixels outside are set to -1 (ignored).
      5) Compute IoU, F1, precision, recall, and average precision using your original metric function.
    """
    preds_batches, targets_batches = [], []

    for input_batch, _ in dataloader:
        input_batch = {k: v.to(device).to(torch.float32) for k, v in input_batch.items()}

        # 1) Prepare the initial T frames, future per-year truth (K frames), static elevation, and base-year map.
        init_def_seq, future_truth_seq, elevation, base_year_map = split_inputs_for_multi_year(
            input_batch=input_batch, T=T, K=K, device=device
        )

        # 2) Autoregressive prediction: cumulative binary prediction (1 if deforested in any of the next K years).
        pred_within_period = predict_deforestation_within_k_years(
            model=model,
            init_def_seq=init_def_seq,
            elevation=elevation,
            K=K,
            device=device,
            binarize_feedback=binarize_feedback,
            fixed_window=fixed_window,
        )  # (B,H,W) in {0,1}

        # 3) Build cumulative ground truth and precise validity mask using the full future frames.
        #    We need the raw statuses (1/0/-1) for each future year, not just the boolean event maps.
        future_frames = input_batch['deforestation_map'][:, T:T+K, 0]  # (B,K,H,W) in {1,0,-1}

        event_seq     = (future_frames == 1)     # (B,K,H,W) True only in the first deforestation year
        nonforest_seq = (future_frames == -1)    # (B,K,H,W)

        # Has an event happened by (and including) each year? (cumulative over time)
        cum_event = torch.cumsum(event_seq.to(torch.int8), dim=1).clamp(max=1).bool()  # (B,K,H,W)

        # Years strictly before the first event (or all years if no event occurs within K)
        pre_event = torch.cat([torch.zeros_like(cum_event[:, :1]), cum_event[:, :-1]], dim=1) == 0  # (B,K,H,W)

        # Invalid if any -1 appears before the first event (or in any year if no event occurs within K)
        invalid_before_event = (nonforest_seq & pre_event).any(dim=1)  # (B,H,W)

        # Base-year eligibility: evaluate only where t0 is forest
        base_forest_mask = (base_year_map == 0)  # (B,H,W)

        # Final validity mask
        valid_mask = base_forest_mask & (~invalid_before_event)  # (B,H,W)

        # Cumulative ground truth: deforested in any of the next K years
        target_within_period = event_seq.any(dim=1).to(torch.int16)  # (B,H,W) {0,1}

        # Apply ignore (-1) outside the valid region
        targets_masked = torch.where(
            valid_mask,
            target_within_period,
            torch.full_like(target_within_period, -1, dtype=torch.int16)
        )

        preds_batches.append(pred_within_period.detach().cpu())
        targets_batches.append(targets_masked.detach().cpu())

    preds   = torch.cat(preds_batches, dim=0)
    targets = torch.cat(targets_batches, dim=0)

    metrics = compute_metrics(targets=targets, preds=preds, dataset_name=f'{dataset_name}_by_{K}y')


    return metrics, preds, targets


In [11]:
device = 'cuda'

config_path = '../config/sample_config.json'
config = config_helper(config_path)
config['annotation_file'] = f"../{config['annotation_file']}"
config['data']['deforestation_map'] = f"../{config['data']['deforestation_map']}"
config['data']['elevation'] = f"../{config['data']['elevation']}"
config['inputs'] = ['deforestation_map', 'elevation']

val_dl = get_dataloaders(['val'], config)['val']

df = pd.read_csv(config['annotation_file'])
df = df.loc[df.split == 'test']
df['split'] = 'val'
df.to_csv("../data/test_annotation.csv", index=False)

config['annotation_file'] = "../data/test_annotation.csv"

device = 'cpu'
config['wandb']['log_to_wandb'] = False
model_input_sequence_len = 4

K_steps = 1


start_year = 2012
end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

_config = config.copy()
_config['val_period'] = {'year_min': start_year, 'year_max': end_year-1, 'input_seq_len': 4}
single_year_dtl = get_dataloaders(['val'], _config)['val']   

# I need to do this to have only samples that have positives in 2012-2016, not 2012-2017 as is done with the multi-year dataloader
dataloader.dataset.annotation = single_year_dtl.dataset.annotation



In [8]:
model = ExponentialDecayBaseline(decay_distance=3.)

multiyear_pred_metrics = {}
multiyear_pred_metrics['baseline'] = {}
for K_steps in [1,2,3,4,5]:


    start_year = start_year
    end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

    config['val_period'] = {'year_min': start_year, 'year_max': end_year, 'input_seq_len': model_input_sequence_len+K_steps}
    dataloader = get_dataloaders(['val'], config)['val']   
    dataloader.dataset.annotation = single_year_dtl.dataset.annotation

    metrics, preds, targets = evaluate_deforestation_within_k_years(
        model,
        dataloader,
        device,
        model_input_sequence_len,
        K_steps)

    print(end_year -2)
    print((targets == 1).sum())
    print(targets.shape)
    print(metrics)
    multiyear_pred_metrics['baseline'][end_year -2] = metrics
    

2016
tensor(237729)
torch.Size([5036, 50, 50])
{'val_by_1y_iou': tensor(0.2250), 'val_by_1y_f1': tensor(0.3674), 'val_by_1y_precision': tensor(0.3350), 'val_by_1y_recall': tensor(0.4067), 'val_by_1y_average_precision': 0.15973753123247605, 'val_by_1y_ROC_AUC': 0.6866848982462741, 'val_by_1y_MAE': tensor(42.3274), 'val_by_1y_MSE': tensor(8776.1699), 'val_by_1y_PearsonR': tensor(0.4903), 'val_by_1y_R2': tensor(0.2198), 'val_by_1y_RMSE': tensor(93.6812)}
2017
tensor(416006)
torch.Size([5036, 50, 50])
{'val_by_2y_iou': tensor(0.2452), 'val_by_2y_f1': tensor(0.3938), 'val_by_2y_precision': tensor(0.3287), 'val_by_2y_recall': tensor(0.4911), 'val_by_2y_average_precision': 0.19672432882923885, 'val_by_2y_ROC_AUC': 0.7081431523449784, 'val_by_2y_MAE': tensor(83.2785), 'val_by_2y_MSE': tensor(20609.4668), 'val_by_2y_PearsonR': tensor(0.5428), 'val_by_2y_R2': tensor(0.1564), 'val_by_2y_RMSE': tensor(143.5600)}
2018
tensor(594481)
torch.Size([5036, 50, 50])
{'val_by_3y_iou': tensor(0.2511), 'val_

In [10]:
import torch
from typing import Dict, Tuple
from torchmetrics import MetricCollection, MeanAbsoluteError, MeanSquaredError, R2Score, PearsonCorrCoef
from torchmetrics.functional.classification import jaccard_index, f1_score, precision, recall
from sklearn.metrics import average_precision_score, roc_auc_score


def split_inputs_for_multi_year(
    input_batch: Dict[str, torch.Tensor],
    T: int,
    K: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Unchanged: prepares inputs and ground truth. See your original docstring.
    """
    def_seq_all = input_batch['deforestation_map'].to(device).to(torch.float32)  # (B, T+K, 1, H, W)
    elevation   = input_batch['elevation'].to(device).to(torch.float32)         # (B, 1, H, W)
    B, X, _, H, W = def_seq_all.shape
    assert X >= T + K, f"Expected at least T+K frames, got X={X} (T={T}, K={K})."

    init_def_seq = def_seq_all[:, :T]                         # (B, T, 1, H, W)
    future_frames = def_seq_all[:, T:T+K, 0]                  # (B, K, H, W)
    future_truth_seq = (future_frames == 1).permute(1, 0, 2, 3)  # (K, B, H, W) bool
    base_year_map = init_def_seq[:, -1, 0]                    # (B, H, W)

    return init_def_seq, future_truth_seq, elevation, base_year_map


@torch.no_grad()
def predict_deforestation_within_k_years(
    model: torch.nn.Module,
    init_def_seq: torch.Tensor,
    elevation: torch.Tensor,
    K: int,
    device: torch.device,
    binarize_feedback: bool = True,
    fixed_window: bool = True
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Modified: returns BOTH the cumulative prediction and the per-year event sequence.

    Returns
    -------
    any_event_within_period : Tensor (B, H, W), float32 in {0,1}
    pred_event_seq          : Tensor (B, K, H, W), bool
        True where a pixel is newly marked as deforested at that step.
    """
    def_seq   = init_def_seq.clone().to(device).to(torch.float32)  # (B, T, 1, H, W)
    elevation = elevation.clone().to(device).to(torch.float32)     # (B, 1, H, W)
    B, T, _, H, W = def_seq.shape

    base_year_map = def_seq[:, -1, 0]                               # (B, H, W)
    ineligible_mask = (base_year_map == -1) | (base_year_map == 1)  # bool

    any_event_within_period = torch.zeros((B, H, W), device=device, dtype=torch.bool)
    pred_event_seq = torch.zeros((B, K, H, W), device=device, dtype=torch.bool)

    for step in range(K):
        # Predict probabilities for the "next year" relative to the current sequence.
        step_probs = model.predict(def_seq[:, -1].squeeze())

        #logits = model({'deforestation_map': def_seq, 'elevation': elevation})  # (B, T_current, 1, H, W)
        #step_probs = torch.sigmoid(logits[:, -1, 0])                            # (B, H, W), values in [0,1]


        forest_mask   = ~ineligible_mask
        new_event_mask = (step_probs >= 0.5) & forest_mask  # bool

        any_event_within_period |= new_event_mask
        pred_event_seq[:, step] = new_event_mask

        ineligible_mask |= new_event_mask

        if binarize_feedback:
            next_frame = torch.zeros_like(base_year_map, dtype=torch.float32)
            next_frame[ineligible_mask] = -1.0
            next_frame[new_event_mask]  = 1.0
        else:
            next_frame = step_probs.clone()
            next_frame[ineligible_mask] = -1.0

        next_frame_t = next_frame.unsqueeze(1).unsqueeze(1)  # (B,1,1,H,W)
        if fixed_window:
            def_seq = torch.cat([def_seq[:, 1:], next_frame_t], dim=1)
        else:
            def_seq = torch.cat([def_seq, next_frame_t], dim=1)

    return any_event_within_period.float(), pred_event_seq  # (B,H,W), (B,K,H,W)


def compute_metrics(targets, preds, dataset_name, regression_metrics=True):
    """
    Unchanged: computes IoU/F1/Precision/Recall/AP/ROC and optional regression metrics.
    """
    preds = preds.to(dtype=torch.float16)
    targets = targets.to(dtype=torch.int16)

    mask = targets != -1
    valid_targets = targets[mask].flatten()
    valid_preds = preds[mask].flatten()

    if regression_metrics:
        binarized_preds = (preds >= 0.5).int()
        pred_pos = ((binarized_preds == 1) & mask).sum(dim=(1,2)).float()
        targ_pos = ((targets == 1) & mask).sum(dim=(1,2)).float()
        has_valid_data = mask.sum(dim=(1,2)) > 0
        pred_pos, targ_pos = pred_pos[has_valid_data], targ_pos[has_valid_data]
        pred_pos = pred_pos.unsqueeze(1)
        targ_pos = targ_pos.unsqueeze(1)
        reg_metrics_coll = MetricCollection({
            f'{dataset_name}_MAE': MeanAbsoluteError(),
            f'{dataset_name}_MSE': MeanSquaredError(),
            f'{dataset_name}_RMSE': MeanSquaredError(squared=False),
            f'{dataset_name}_R2': R2Score(),
            f'{dataset_name}_PearsonR': PearsonCorrCoef(),
        })
        reg_metrics = reg_metrics_coll(pred_pos, targ_pos)

    metrics = {
        f'{dataset_name}_iou' : jaccard_index(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_f1' : f1_score(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_precision' : precision(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_recall' : recall(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_average_precision' : average_precision_score(valid_targets.flatten(), valid_preds.flatten()) if valid_targets.numel() > 0 else float('nan'),
        f'{dataset_name}_ROC_AUC' : roc_auc_score(valid_targets.flatten(), valid_preds.flatten()) if valid_targets.numel() > 0 else float('nan'),
    }
    if regression_metrics:
        metrics.update(reg_metrics)
    return metrics


@torch.no_grad()
def evaluate_deforestation_within_k_years(
    model: torch.nn.Module,
    dataloader,
    device: torch.device,
    T: int,
    K: int,
    binarize_feedback: bool = True,
    fixed_window: bool = True,
    dataset_name: str = 'val',
    log_to_wandb: bool = False,
) -> Dict[str, torch.Tensor]:
    """
    Modified: returns metrics for ALL temporal tolerances τ ∈ {0,...,K-1} in one run.

    For each τ:
      - TP: predicted and true event exist and |t_pred - t_true| ≤ τ
      - FP: predicted event exists but not a τ-match (includes no truth OR out-of-window)
      - FN: true event exists but not a τ-match
      - TN: neither predicted nor true event
    Implemented by encoding τ into the predicted map before calling `compute_metrics`.
    """
    model.eval()
    # Accumulate per-τ preds and targets
    preds_per_tau = [ [] for _ in range(K) ]   # each element: list of (B_batch,H,W) tensors
    targs_per_tau = [ [] for _ in range(K) ]

    for input_batch, _ in dataloader:
        input_batch = {k: v.to(device).to(torch.float32) for k, v in input_batch.items()}

        # 1) Prepare inputs
        init_def_seq, future_truth_seq, elevation, base_year_map = split_inputs_for_multi_year(
            input_batch=input_batch, T=T, K=K, device=device
        )  # future_truth_seq: (K,B,H,W) bool
        truth_seq = future_truth_seq.permute(1,0,2,3).contiguous()  # (B,K,H,W) bool

        # 2) Predict per-year events + cumulative
        pred_within_period, pred_seq = predict_deforestation_within_k_years(
            model=model,
            init_def_seq=init_def_seq,
            elevation=elevation,
            K=K,
            device=device,
            binarize_feedback=binarize_feedback,
            fixed_window=fixed_window,
        )  # pred_within_period:(B,H,W), pred_seq:(B,K,H,W) bool

        # 3) Build validity mask using raw future frames
        future_frames = input_batch['deforestation_map'][:, T:T+K, 0]  # (B,K,H,W) in {1,0,-1}
        event_seq     = (future_frames == 1)     # (B,K,H,W)
        nonforest_seq = (future_frames == -1)    # (B,K,H,W)

        cum_event = torch.cumsum(event_seq.to(torch.int8), dim=1).clamp(max=1).bool()  # (B,K,H,W)
        pre_event = torch.cat([torch.zeros_like(cum_event[:, :1]), cum_event[:, :-1]], dim=1) == 0
        invalid_before_event = (nonforest_seq & pre_event).any(dim=1)  # (B,H,W)
        base_forest_mask = (base_year_map == 0)  # (B,H,W)
        valid_mask = base_forest_mask & (~invalid_before_event)  # (B,H,W)

        # 4) First-event year indices for truth and prediction
        #    Find first year where event=True; if none, assign sentinel K.
        # Truth
        truth_any = truth_seq.any(dim=1)  # (B,H,W)
        t_true = torch.full((truth_seq.shape[0], truth_seq.shape[2], truth_seq.shape[3]),
                            K, device=truth_seq.device, dtype=torch.int64)
        if truth_any.any():
            # Get indices of first True per pixel
            first_true_idx = torch.argmax(truth_seq.to(torch.int8), dim=1)
            # Mask out pixels with no event
            t_true = torch.where(truth_any, first_true_idx, t_true)

        # Prediction
        pred_any = pred_seq.any(dim=1)
        t_pred = torch.full((pred_seq.shape[0], pred_seq.shape[2], pred_seq.shape[3]),
                            K, device=pred_seq.device, dtype=torch.int64)
        if pred_any.any():
            first_pred_idx = torch.argmax(pred_seq.to(torch.int8), dim=1)
            t_pred = torch.where(pred_any, first_pred_idx, t_pred)

        # 5) Per-τ targets and preds encoding
        #    Targets are "has true event" (independent of τ).
        targets_base = truth_any.to(torch.int16)  # (B,H,W) in {0,1}
        #    For preds, mark 1 if:
        #      - no true event (so any prediction is FP and must remain 1)
        #      - OR prediction exists AND |t_pred - t_true| ≤ τ  (counts as TP)
        absdiff = torch.abs(t_pred.to(torch.int32) - t_true.to(torch.int32))  # (B,H,W), large when any=K

        for tau in range(K):
            match = pred_any & truth_any & (absdiff <= tau)       # pixels that are TP under this τ
            preds_tau = (pred_any & (~truth_any | match)).to(torch.float32)  # FP kept, TP kept, FN -> 0
            targets_tau = targets_base.clone()

            # Apply validity mask: outside → -1 on targets; preds value ignored by metrics via ignore_index
            targets_masked = torch.where(
                valid_mask,
                targets_tau,
                torch.full_like(targets_tau, -1, dtype=torch.int16)
            )

            preds_per_tau[tau].append(preds_tau.detach().cpu())
            targs_per_tau[tau].append(targets_masked.detach().cpu())

    # 6) Aggregate and compute metrics for each τ
    all_metrics = {}
    for tau in range(K):
        preds   = torch.cat(preds_per_tau[tau], dim=0)   # (N,H,W)
        targets = torch.cat(targs_per_tau[tau], dim=0)   # (N,H,W)
        metrics_tau = compute_metrics(
            targets=targets,
            preds=preds,
            dataset_name=f'{dataset_name}_by_{K}y'
        )
        
        for k,v in metrics_tau.items():
            try:
                metrics_tau[k] = round(v.item(), 4)
            except:
                pass
            
        # Store metrics under nested dictionary keyed by tau
        all_metrics[tau] = metrics_tau

    return all_metrics


In [14]:
device  = 'cpu'

In [16]:
model = ExponentialDecayBaseline(decay_distance=3.)

multiyear_pred_metrics = {}
multiyear_pred_metrics['baseline'] = {}
for K_steps in [1,2,3,4,5]:

    model_input_sequence_len = model_input_sequence_len

    start_year = start_year
    end_year = start_year + model_input_sequence_len + K_steps + 1 # adding +1 because last item not included

    config['val_period'] = {'year_min': start_year, 'year_max': end_year, 'input_seq_len': model_input_sequence_len+K_steps}
    dataloader = get_dataloaders(['val'], config)['val']   
    dataloader.dataset.annotation = single_year_dtl.dataset.annotation

    metrics = evaluate_deforestation_within_k_years(
        model,
        dataloader,
        device,
        model_input_sequence_len,
        K_steps)

    print(end_year -2)
    print(metrics)
    multiyear_pred_metrics['baseline'][end_year -2] = metrics
    

2016
{0: {'val_by_1y_iou': 0.225, 'val_by_1y_f1': 0.3674, 'val_by_1y_precision': 0.335, 'val_by_1y_recall': 0.4067, 'val_by_1y_average_precision': 0.1597, 'val_by_1y_ROC_AUC': 0.6867, 'val_by_1y_MAE': 42.3274, 'val_by_1y_MSE': 8776.1699, 'val_by_1y_PearsonR': 0.4903, 'val_by_1y_R2': 0.2198, 'val_by_1y_RMSE': 93.6812}}
2017
{0: {'val_by_2y_iou': 0.1536, 'val_by_2y_f1': 0.2663, 'val_by_2y_precision': 0.2347, 'val_by_2y_recall': 0.3076, 'val_by_2y_average_precision': 0.1203, 'val_by_2y_ROC_AUC': 0.6164, 'val_by_2y_MAE': 79.5439, 'val_by_2y_MSE': 21311.7305, 'val_by_2y_PearsonR': 0.4564, 'val_by_2y_R2': 0.1276, 'val_by_2y_RMSE': 145.9854}, 1: {'val_by_2y_iou': 0.2452, 'val_by_2y_f1': 0.3938, 'val_by_2y_precision': 0.3287, 'val_by_2y_recall': 0.4911, 'val_by_2y_average_precision': 0.1967, 'val_by_2y_ROC_AUC': 0.7081, 'val_by_2y_MAE': 83.2785, 'val_by_2y_MSE': 20609.4668, 'val_by_2y_PearsonR': 0.5428, 'val_by_2y_R2': 0.1564, 'val_by_2y_RMSE': 143.56}}
2018
{0: {'val_by_3y_iou': 0.1195, 'val_

In [19]:
multiyear_pred_metrics

{'baseline': {2016: {0: {'val_by_1y_iou': 0.225,
    'val_by_1y_f1': 0.3674,
    'val_by_1y_precision': 0.335,
    'val_by_1y_recall': 0.4067,
    'val_by_1y_average_precision': 0.1597,
    'val_by_1y_ROC_AUC': 0.6867,
    'val_by_1y_MAE': 42.3274,
    'val_by_1y_MSE': 8776.1699,
    'val_by_1y_PearsonR': 0.4903,
    'val_by_1y_R2': 0.2198,
    'val_by_1y_RMSE': 93.6812}},
  2017: {0: {'val_by_2y_iou': 0.1536,
    'val_by_2y_f1': 0.2663,
    'val_by_2y_precision': 0.2347,
    'val_by_2y_recall': 0.3076,
    'val_by_2y_average_precision': 0.1203,
    'val_by_2y_ROC_AUC': 0.6164,
    'val_by_2y_MAE': 79.5439,
    'val_by_2y_MSE': 21311.7305,
    'val_by_2y_PearsonR': 0.4564,
    'val_by_2y_R2': 0.1276,
    'val_by_2y_RMSE': 145.9854},
   1: {'val_by_2y_iou': 0.2452,
    'val_by_2y_f1': 0.3938,
    'val_by_2y_precision': 0.3287,
    'val_by_2y_recall': 0.4911,
    'val_by_2y_average_precision': 0.1967,
    'val_by_2y_ROC_AUC': 0.7081,
    'val_by_2y_MAE': 83.2785,
    'val_by_2y_MSE': 206